# Category + subcategory pair prediction
Frozen sentence embeddings + logistic regression. **Grouped evaluation only: five outer test folds and three inner selection folds.** One class represents one valid category/subcategory pair.

Inputs: prepared initial short description + description. Labels: reviewed category and subcategory. No category, priority or escalation outcome is fed as an input. Identical text and incident/problem-linked tickets stay together through connected grouping.

The synthetic dataset contains repeated templates. High scores here demonstrate a working pipeline, not production accuracy or a verified improvement over the current routing system.

In [1]:
from pathlib import Path
import sys, subprocess
import pandas as pd
from IPython.display import display
candidates = [Path.cwd(), Path.cwd() / 'category_pair_model', Path.cwd() / 'ticket_data_preparation/category_pair_model']
MODEL_DIR = next((p for p in candidates if (p / 'train_category_pair.py').exists()), None)
if MODEL_DIR is None:
    raise FileNotFoundError('Open the notebook from the extracted ticket_data_preparation folder.')
print('Model folder:', MODEL_DIR)


Model folder: /workspace/scratch/ec4ca456f2d2/project/ticket_data_preparation/category_pair_model


## Train and evaluate
Run the shared training script. C is selected by inner out-of-fold pair macro F1; each outer test fold is evaluated once. Scores choose the highest-probability pair; the escalation recall threshold does not apply. The final deployable model is fitted separately after evaluation.

Rare labels may have fewer examples than the fold count. Their held-out predictions are retained; see pair support and unseen-pair counts. Stratification is approximate with intact groups.

In [2]:
completed = subprocess.run([sys.executable, str(MODEL_DIR / 'train_category_pair.py')], capture_output=True, text=True)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
completed.check_returncode()


2625 tickets; 39 pairs; 803 connected groups
Fold 1: C=10.0, pair accuracy=0.9962
Fold 2: C=10.0, pair accuracy=1.0000
Fold 3: C=100.0, pair accuracy=1.0000
Fold 4: C=100.0, pair accuracy=1.0000
Fold 5: C=100.0, pair accuracy=1.0000
                 model  tickets  pair_accuracy  pair_macro_f1  pair_weighted_f1  category_accuracy  category_macro_f1  category_weighted_f1  subcategory_accuracy  subcategory_macro_f1  subcategory_weighted_f1
     embedding_lr_pair     2625         0.9992         0.9995            0.9992              1.000             1.0000                 1.000                0.9992                0.9995                   0.9992
training_majority_pair     2625         0.1032         0.0048            0.0193              0.269             0.0424                 0.114                0.1032                0.0048                   0.0193


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5113.43it/s]
/opt/codex/runtimes/codex-primary-runtime/dependencies/python/lib/pyt

## Held-out grouped results
Exact-pair accuracy requires both outputs to match. Macro F1 gives equal weight to evaluated classes. The majority baseline picks the most common training pair separately in each outer fold.

In [3]:
OUT = MODEL_DIR / 'results'
display(pd.read_csv(OUT / 'grouped_metrics.csv'))
display(pd.read_csv(OUT / 'grouped_fold_metrics.csv'))
display(pd.read_csv(OUT / 'misclassified_pairs.csv')[['ticket_id', 'target_category', 'target_subcategory', 'predicted_category', 'predicted_subcategory']])


model,tickets,pair_accuracy,pair_macro_f1,pair_weighted_f1,category_accuracy,category_macro_f1,category_weighted_f1,subcategory_accuracy,subcategory_macro_f1,subcategory_weighted_f1
embedding_lr_pair,2625,0.999238,0.999542,0.999238,1.000000,1.00000,1.000000,0.999238,0.999542,0.999238
training_majority_pair,2625,0.103238,0.004799,0.019321,0.268952,0.04239,0.114008,0.103238,0.004799,0.019321


fold,selected_C,train_tickets,test_tickets,pair_accuracy,pair_macro_f1,unseen_pair_tickets
1,10.0,2096,529,0.996219,0.997653,0
2,10.0,2087,538,1.000000,1.000000,0
3,100.0,2089,536,1.000000,1.000000,0
4,100.0,2118,507,1.000000,1.000000,0
5,100.0,2110,515,1.000000,1.000000,0


ticket_id,target_category,target_subcategory,predicted_category,predicted_subcategory
TKT0100292,Investor Portal,Document not visible,Investor Portal,Navigation question
TKT0100889,Investor Portal,Document not visible,Investor Portal,Navigation question


## Predict new tickets
Uses the final classifier trained on all eligible rows. These example predictions are demonstrations, not additional test results. Pair confidence is uncalibrated. Use the same initial-text cleaning as training before prediction.

In [4]:
prediction = subprocess.run([sys.executable, str(MODEL_DIR / 'predict_category_pair.py')], capture_output=True, text=True)
print(prediction.stdout)
if prediction.returncode:
    print(prediction.stderr)
prediction.check_returncode()


ticket_id               short_description                                                              description predicted_category     predicted_subcategory  pair_confidence
   NEW001 Instrument data looks incorrect The coupon and maturity for an instrument look incorrect. Please verify.          Analytics  Instrument data question         0.995118
   NEW002       Model assignment question               How can we assign one model to a group of client accounts?             Wealth Model assignment question         0.996970



## Verify grouping and outputs
Tests check intact groups, one held-out prediction per eligible ticket, no train/test group overlap, valid predicted pairs, and final model label coverage. Existing escalation notebooks are separate and unchanged.

In [5]:
checks = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(MODEL_DIR), '-p', 'test_category_pair.py', '-v'], capture_output=True, text=True)
print(checks.stdout + checks.stderr)
checks.check_returncode()


test_group_integrity (test_category_pair.PairTests.test_group_integrity) ... ok
test_held_out_predictions_and_saved_model (test_category_pair.PairTests.test_held_out_predictions_and_saved_model) ... ok
test_pair_roundtrip (test_category_pair.PairTests.test_pair_roundtrip) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.108s

OK

